In [1]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"titanjd","key":"13a24d9a0251f2dbe716b6e90c2431e4"}'}

In [2]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [3]:
#!/bin/bash
!kaggle datasets download jp797498e/twitter-entity-sentiment-analysis

Dataset URL: https://www.kaggle.com/datasets/jp797498e/twitter-entity-sentiment-analysis
License(s): CC0-1.0
100% 1.99M/1.99M [00:00<00:00, 3.39MB/s]



In [4]:
import zipfile
import os

# Specify the name of the downloaded zip file (e.g., 'titanic.zip')
zip_file_name = 'twitter-entity-sentiment-analysis.zip'

# Create a directory for the extracted files
destination_dir = '/content/dataset_files'
os.makedirs(destination_dir, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
    zip_ref.extractall(destination_dir)

# (Optional) Remove the zip file after extraction
os.remove(zip_file_name)

# List the extracted files
print(f"Extracted files: {os.listdir(destination_dir)}")

Extracted files: ['twitter_training.csv', 'twitter_validation.csv']


In [5]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from torch import nn
import numpy as np
import re
from collections import Counter

# Load the datasets
train_df = pd.read_csv('/content/dataset_files/twitter_training.csv')
validation_df = pd.read_csv('/content/dataset_files/twitter_validation.csv')

# Rename columns for consistency and drop unnecessary ones
train_df.columns = ['Tweet_ID', 'Entity', 'Sentiment', 'Tweet_Text']
validation_df.columns = ['Tweet_ID', 'Entity', 'Sentiment', 'Tweet_Text']

# Drop rows with NaN values in 'Tweet_Text' as they are unusable
train_df.dropna(subset=['Tweet_Text'], inplace=True)
validation_df.dropna(subset=['Tweet_Text'], inplace=True)

# Map sentiments to numerical labels (0, 1, 2, 3)
sentiment_mapping = {'Positive': 0, 'Negative': 1, 'Neutral': 2, 'Irrelevant': 3}
train_df['Sentiment_Label'] = train_df['Sentiment'].map(sentiment_mapping)
validation_df['Sentiment_Label'] = validation_df['Sentiment'].map(sentiment_mapping)

# Drop rows where Sentiment_Label is NaN (if any unmapped sentiments exist)
train_df.dropna(subset=['Sentiment_Label'], inplace=True)
validation_df.dropna(subset=['Sentiment_Label'], inplace=True)

# Convert labels to integers
train_df['Sentiment_Label'] = train_df['Sentiment_Label'].astype(int)
validation_df['Sentiment_Label'] = validation_df['Sentiment_Label'].astype(int)

print("Train Data Head:")
display(train_df.head())
print("Validation Data Head:")
display(validation_df.head())

# Determine OUTPUT_SIZE based on unique sentiment labels
OUTPUT_SIZE = train_df['Sentiment_Label'].nunique()
print(f"Output size (number of classes): {OUTPUT_SIZE}")

Train Data Head:


,Tweet_ID,Entity,Sentiment,Tweet_Text,Sentiment_Label
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...,0
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...,0
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...,0
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...,0
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...,0


Validation Data Head:


,Tweet_ID,Entity,Sentiment,Tweet_Text,Sentiment_Label
0,352,Amazon,Neutral,BBC News - Amazon boss Jeff Bezos rejects clai...,2
1,8312,Microsoft,Negative,@Microsoft Why do I pay for WORD when it funct...,1
2,4371,CS-GO,Negative,"CSGO matchmaking is so full of closet hacking,...",1
3,4433,Google,Neutral,Now the President is slapping Americans in the...,2
4,6273,FIFA,Negative,Hi @EAHelp I’ve had Madeleine McCann in my cel...,1


Output size (number of classes): 4


In [6]:
# Text preprocessing function
def preprocess_text(text):
    text = str(text).lower() # Convert to lowercase and ensure it's a string
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Remove URLs
    text = re.sub(r'<.*?>', '', text) # Remove HTML tags
    text = re.sub(r'[^a-z0-9\s]', '', text) # Remove punctuation and special characters
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
    return text

train_df['Tweet_Text'] = train_df['Tweet_Text'].apply(preprocess_text)
validation_df['Tweet_Text'] = validation_df['Tweet_Text'].apply(preprocess_text)

print("Preprocessed Train Data Head:")
display(train_df.head())
print("Preprocessed Validation Data Head:")
display(validation_df.head())

Preprocessed Train Data Head:


,Tweet_ID,Entity,Sentiment,Tweet_Text,Sentiment_Label
0,2401,Borderlands,Positive,i am coming to the borders and i will kill you...,0
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you all,0
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...,0
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...,0
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...,0


Preprocessed Validation Data Head:


,Tweet_ID,Entity,Sentiment,Tweet_Text,Sentiment_Label
0,352,Amazon,Neutral,bbc news amazon boss jeff bezos rejects claims...,2
1,8312,Microsoft,Negative,microsoft why do i pay for word when it functi...,1
2,4371,CS-GO,Negative,csgo matchmaking is so full of closet hacking ...,1
3,4433,Google,Neutral,now the president is slapping americans in the...,2
4,6273,FIFA,Negative,hi eahelp ive had madeleine mccann in my cella...,1


In [7]:
# Tokenization and Vocabulary Building
def tokenize_text(df, text_col='Tweet_Text'):
    all_words = ' '.join(df[text_col]).split()
    return all_words

train_words = tokenize_text(train_df)

# Build vocabulary and word-to-index mapping
word_counts = Counter(train_words)
vocabulary = sorted(word_counts, key=word_counts.get, reverse=True)
word_to_idx = {word: idx + 2 for idx, word in enumerate(vocabulary)}
word_to_idx['<pad>'] = 0
word_to_idx['<unk>'] = 1

idx_to_word = {idx: word for word, idx in word_to_idx.items()}
vocab_size = len(word_to_idx)

print(f"Vocabulary size: {vocab_size}")
print("Top 10 words in vocabulary:", vocabulary[:10])

Vocabulary size: 39190
Top 10 words in vocabulary: ['the', 'i', 'to', 'and', 'a', 'of', 'is', 'for', 'in', 'this']


In [8]:
# Convert text to sequences and pad
def text_to_sequence(text, word_to_idx, max_len):
    sequence = [word_to_idx.get(word, word_to_idx['<unk>']) for word in text.split()]
    if len(sequence) > max_len:
        sequence = sequence[:max_len]
    else:
        sequence = [word_to_idx['<pad>']] * (max_len - len(sequence)) + sequence
    return sequence

MAX_SEQUENCE_LENGTH = 50

train_sequences = [text_to_sequence(text, word_to_idx, MAX_SEQUENCE_LENGTH) for text in train_df['Tweet_Text']]
validation_sequences = [text_to_sequence(text, word_to_idx, MAX_SEQUENCE_LENGTH) for text in validation_df['Tweet_Text']]

train_sequences = np.array(train_sequences)
validation_sequences = np.array(validation_sequences)

print(f"Shape of training sequences: {train_sequences.shape}")
print(f"Shape of validation sequences: {validation_sequences.shape}")
print("First training sequence:", train_sequences[0])
print("Corresponding text:", train_df['Tweet_Text'].iloc[0])

Shape of training sequences: (73995, 50)
Shape of validation sequences: (999, 50)
First training sequence: [   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    3  111  387    4
    2 6511    5    3   51  417   13   27]
Corresponding text: i am coming to the borders and i will kill you all


In [9]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, output_size):
        super(LSTMModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True) # Changed to LSTM
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden_state, cell_state) = self.lstm(embedded)
        last_hidden_state = hidden_state.squeeze(0)
        output = self.fc(last_hidden_state)
        return output

EMBEDDING_DIM = 100
HIDDEN_SIZE = 128
OUTPUT_SIZE = 4

model = LSTMModel(vocab_size, EMBEDDING_DIM, HIDDEN_SIZE, OUTPUT_SIZE)

print(model)

LSTMModel(
  (embedding): Embedding(39190, 100)
  (lstm): LSTM(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=4, bias=True)
)


In [10]:
class TweetDataset(Dataset):
    def __init__(self, sequences, labels=None):
        self.sequences = torch.LongTensor(sequences)
        self.labels = torch.LongTensor(labels) if labels is not None else None

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        if self.labels is not None:
            return self.sequences[idx], self.labels[idx]
        return self.sequences[idx]

BATCH_SIZE = 64

train_labels = train_df['Sentiment_Label'].values
train_dataset = TweetDataset(train_sequences, train_labels)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

validation_labels = validation_df['Sentiment_Label'].values
validation_dataset = TweetDataset(validation_sequences, validation_labels)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Number of batches in train_loader: {len(train_loader)}")
print(f"Number of batches in validation_loader: {len(validation_loader)}")

Number of batches in train_loader: 1157
Number of batches in validation_loader: 16


In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print(f"Using device: {device}")

Using device: cuda


In [12]:
NUM_EPOCHS = 5

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    correct_predictions = 0
    total_samples = 0

    for batch_idx, (sequences, labels) in enumerate(train_loader):
        sequences, labels = sequences.to(device), labels.to(device)

        # Forward pass
        outputs = model(sequences)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / total_samples

    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}], Training Loss: {avg_loss:.4f}, Training Accuracy: {accuracy:.4f}')

    model.eval()
    val_correct_predictions = 0
    val_total_samples = 0
    val_total_loss = 0
    with torch.no_grad():
        for sequences, labels in validation_loader:
            sequences, labels = sequences.to(device), labels.to(device)
            outputs = model(sequences)
            loss = criterion(outputs, labels)
            val_total_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            val_total_samples += labels.size(0)
            val_correct_predictions += (predicted == labels).sum().item()

    val_avg_loss = val_total_loss / len(validation_loader)
    val_accuracy = val_correct_predictions / val_total_samples
    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}], Validation Loss: {val_avg_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}')

Epoch [1/5], Training Loss: 1.0764, Training Accuracy: 0.5426
Epoch [1/5], Validation Loss: 0.7218, Validation Accuracy: 0.7167
Epoch [2/5], Training Loss: 0.6344, Training Accuracy: 0.7587
Epoch [2/5], Validation Loss: 0.4025, Validation Accuracy: 0.8749
Epoch [3/5], Training Loss: 0.3400, Training Accuracy: 0.8779
Epoch [3/5], Validation Loss: 0.2445, Validation Accuracy: 0.9239
Epoch [4/5], Training Loss: 0.1979, Training Accuracy: 0.9293
Epoch [4/5], Validation Loss: 0.2422, Validation Accuracy: 0.9429
Epoch [5/5], Training Loss: 0.1355, Training Accuracy: 0.9500
Epoch [5/5], Validation Loss: 0.2523, Validation Accuracy: 0.9499
